# 03 - Policy Training

Run the deterministic offline policy comparison and write the selected policy artifacts under `reports/policy_training/`.

Cosmos DB publication is a later audit/reuse step. The training command itself writes local artifacts only; publication to the `policy_versions` container happens after validation.

## Compared Policies

- `baseline`
- `epsilon_greedy`
- `ucb`
- `thompson_sampling`

The runner compares policies on the same processed Hillstrom sequence and selects the strongest local candidate using reward, regret, and exploration metrics.


In [ ]:
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

processed_file = repo_root / 'data' / 'processed' / 'hillstrom_processed.csv'
report_dir = repo_root / 'reports' / 'policy_training'

{
    'processed_file_exists': processed_file.exists(),
    'report_dir': str(report_dir),
    'report_dir_exists': report_dir.exists(),
}

## Run Training

Use the capped command for fast local review:

```bash
python -m src.evaluation.run --max-rows 5000
```

Use the full command when you need the complete local report:

```bash
python -m src.evaluation.run
```

In [ ]:
# Optional execution cell. Uncomment after processing and validation are complete.
# import os
# os.chdir(repo_root)
# !python -m src.evaluation.run --max-rows 5000

## Post-Training Artifact Validation

Run artifact validation after training and before any Blob or Cosmos publication. The validator checks metrics, selected policy, Golden Set, purchase-likelihood model, data-validation status, and checksums, then writes `artifact_manifest.json`.


In [ ]:
# Optional execution cell. Uncomment after training completes.
# !python -m src.evaluation.validate_artifacts


## Selected Policy

In [ ]:
import json

selected_path = report_dir / 'selected_policy.json'
if selected_path.exists():
    json.loads(selected_path.read_text(encoding='utf-8'))
else:
    {'status': 'missing', 'next_step': 'Run python -m src.evaluation.run.'}

## Generated Artifacts

In [ ]:
expected_artifacts = [
    'metrics.json',
    'metrics.csv',
    'policy_versions.json',
    'selected_policy.json',
    'golden_set_recommendations.json',
    'policy_state_thompson_sampling.json',
    'purchase_likelihood_model.json',
    'artifact_manifest.json',
]

[
    {
        'artifact': name,
        'exists': (report_dir / name).exists(),
        'size_kb': round((report_dir / name).stat().st_size / 1000, 1) if (report_dir / name).exists() else None,
    }
    for name in expected_artifacts
]
